<div style="margin-bottom: 1em;">
  <a href="https://colab.research.google.com/github/google-deepmind/jax_privacy/blob/main/examples/dp_sgd_keras_gemma3_dpsapf.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" />
  </a>
  <a href="https://github.com/google-deepmind/jax_privacy/blob/main/examples/dp_sgd_keras_gemma3_dpsapf.ipynb" target="_blank" style="margin-left: 10px;">
    <img src="https://img.shields.io/badge/GitHub-view--source-black?logo=github" />
  </a>
</div>

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# https://www.apache.org/licenses/LICENSE-2.0

# Auto-LoRA + DP-SGD fine-tuning of Gemma3

**Copyright 2026 DeepMind Technologies Limited.**

This notebook adapts **DP-SAPF** (*Saliency-Aware Parameter Fine-tuning of Public Models for Differentially Private Image Synthesis*; [arXiv:2605.30312](https://arxiv.org/abs/2605.30312)) to LLM fine-tuning. The core idea of DP-SAPF: under DP-SGD, the noise added to every trainable parameter scales with the parameter count, so adapting **every** layer wastes privacy budget on uninformative directions. Instead, first run a **DP-aware saliency probe** to identify which layers carry the most useful gradient signal for the task, then fine-tune only that small subset — getting a better signal-to-noise ratio under the same (eps, delta) budget.

Concretely, this notebook extends [`dp_sgd_keras_gemma3_lora_finetuning_samsum.ipynb`](dp_sgd_keras_gemma3_lora_finetuning_samsum.ipynb) with two new pieces:

1. **Probe (DP)**: for each training sample, vote +1 on the top-k LoRA-candidate layers ranked by per-sample gradient L2 norm; add Gaussian noise to the vote histogram (L2 sensitivity = $\sqrt k$).
2. **DP-SGD fine-tune**: enable LoRA only on the top-k% layers by score; calibrate the training noise so the **composed** (probe + DP-SGD) cost matches a single target $\varepsilon$.

Selection-specific code (the probe, top-k voting, LoRA gating, DP composition) lives in [`dpsapf_utils.py`](https://github.com/google-deepmind/jax_privacy/tree/main/examples/dpsapf_utils.py); everything else (data loading, model setup, optimizer, fit, eval) stays inline so the moving parts are visible.

## Install and configure

In [ ]:
%%capture
!pip install -q -U "keras>=3" keras-nlp keras-hub rouge-score tqdm ipywidgets
!pip install -q dp_accounting jaxtyping drjax tensorflow_datasets datasets
!pip install -q beautifulsoup4 lxml jax_privacy==1.0.0

In [ ]:
import os

# Redirect big artifacts to a shared scratch dir; set env vars BEFORE
# importing keras / kagglehub / datasets.
CACHE_ROOT = "./"  # change to a writable path
os.makedirs(CACHE_ROOT, exist_ok=True)
for k, sub in [
    ("KERAS_HOME", "keras"),
    ("KAGGLEHUB_CACHE", "kagglehub"),
    ("TFDS_DATA_DIR", "tfds"),
    ("HF_HOME", "huggingface"),
    ("JAX_COMPILATION_CACHE_DIR", "jax_compilation_cache"),
]:
  os.environ.setdefault(k, os.path.join(CACHE_ROOT, sub))
  os.makedirs(os.environ[k], exist_ok=True)

os.environ["KERAS_BACKEND"] = "jax"
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.85")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import gc
import keras
import keras_hub
import tensorflow as tf
import tqdm
import jax
from jax_privacy import keras_api

# Selection-only helpers — everything else is inline.
import examples.dpsapf_utils as ALU

## Kaggle login

Needed to download Gemma3 weights. You must **accept the Gemma3 license** in your Kaggle account for each variant ([1B](https://www.kaggle.com/models/keras/gemma3/keras/gemma3_instruct_1b), [4B](https://www.kaggle.com/models/keras/gemma3/keras/gemma3_instruct_4b_text)) or the download returns 403.

In [ ]:
import kagglehub

kagglehub.login()
# Or: export KAGGLE_USERNAME=... KAGGLE_KEY=... before launching the notebook.

## Hyper-parameters

Two datasets: `cnn_dailymail` (TFDS) and `xsum_hf` (HuggingFace). Both ~200-290K examples — well-sized for DP subsampling amplification.

In [ ]:
TEST_RUN = True
DATASET = "cnn_dailymail"  # or "xsum_hf"

# Model / training
GEMMA3_MODEL_TYPE = "gemma3_instruct_4b_text"
SEQUENCE_LENGTH = 1024
TEST_DS_SEQUENCE_LENGTH = 1024
EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 128  # effective batch = 4 * 128 = 512
TEST_DS_BATCH_SIZE = 4
LORA_RANK = 64
LEARNING_RATE = 3e-3
SEED = 0
USE_MIXED_PRECISION = False

# Probe + selection
PROBE_SAMPLES = 10000
PROBE_TOPK = 8
PROBE_NOISE_MULTIPLIER = 40.0  # in units of L2 sensitivity sqrt(topk)
LORA_TOP_K_PERCENT = 10.0  # 1-5%% wins on CNN/DM at 1024+DP
LORA_ATTN_ONLY = True

# Composed DP target
TOTAL_EPSILON = 4.0
DELTA = 2e-5
CLIPPING_NORM = 0.001

if TEST_RUN:
  GEMMA3_MODEL_TYPE = "gemma3_instruct_1b"
  PROBE_SAMPLES = 10000

DATASET_CFG = ALU.DATASET_REGISTRY[DATASET]
print(f"Dataset: {DATASET} ({DATASET_CFG['note']})")
print(f"Model:   {GEMMA3_MODEL_TYPE} (test_run={TEST_RUN})")
print(f"Effective train batch: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

## Data

Load + format using helpers from `dpsapf_utils` (registry + HF/TFDS dispatch + prompt formatter), then shuffle/batch inline.

In [ ]:
fmt = ALU.make_source_to_gemma3_format(DATASET_CFG)
TRAIN_DS = ALU.load_dataset_split(DATASET_CFG, "train").map(fmt)
VALIDATION_DS = ALU.load_dataset_split(DATASET_CFG, "validation").map(fmt)
TEST_DS = ALU.load_dataset_split(DATASET_CFG, "test").map(fmt)

TRAIN_SIZE = int(TRAIN_DS.cardinality().numpy())
print(
    f"Train: {TRAIN_SIZE}  Val: {int(VALIDATION_DS.cardinality().numpy())}  "
    f"Test: {int(TEST_DS.cardinality().numpy())}"
)

TRAIN_DS_BATCHED = TRAIN_DS.shuffle(2048).batch(BATCH_SIZE, drop_remainder=True)
VALIDATION_DS_BATCHED = VALIDATION_DS.batch(BATCH_SIZE, drop_remainder=True)
TEST_DS_BATCHED = TEST_DS.batch(TEST_DS_BATCH_SIZE)

EXAMPLE = (
    VALIDATION_DS.take(1)
    .batch(1, drop_remainder=True)
    .as_numpy_iterator()
    .next()
)
for k, v in EXAMPLE.items():
  s = v[0].decode("utf-8")
  print(f'{k}: "{s[:200]}{"..." if len(s) > 200 else ""}"\n')

## Load Gemma3 (no LoRA yet) and run baseline inference

In [ ]:
MODEL_WEIGHTS_DTYPE = None
if USE_MIXED_PRECISION:
  keras.mixed_precision.set_global_policy("mixed_bfloat16")
  MODEL_WEIGHTS_DTYPE = "bfloat16"

keras.distribution.set_distribution(keras.distribution.DataParallel())

gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset(
    GEMMA3_MODEL_TYPE, dtype=MODEL_WEIGHTS_DTYPE
)
gemma_lm.preprocessor.sequence_length = SEQUENCE_LENGTH


def show_inference():
  print(gemma_lm.generate(EXAMPLE["prompts"])[0])
  print(f"\nGold:\n{EXAMPLE['responses'][0].decode('utf-8')}")


print("=== Baseline (before fine-tuning) ===")
show_inference()

## DP probe + layer selection

Per sample, vote +1 on the top-k LoRA-candidate layers ranked by per-sample gradient L2 norm. Gaussian noise on the vote histogram; L2 sensitivity = $\sqrt k$ under ADD/REMOVE neighboring. `ALU.run_probe` returns the noisy scores, the top-k% selected paths, and DP metadata used downstream for composition.

In [ ]:
probe = ALU.run_probe(
    gemma_lm,
    TRAIN_DS_BATCHED,
    top_k_percent=LORA_TOP_K_PERCENT,
    num_probe_samples=PROBE_SAMPLES,
    topk=PROBE_TOPK,
    noise_multiplier=PROBE_NOISE_MULTIPLIER,
    attn_only=LORA_ATTN_ONLY,
    seed=SEED,
)

## Enable LoRA on selected layers, set up composed DP-SGD

Both the probe and DP-SGD are Poisson-subsampled Gaussian mechanisms. We solve for $\sigma_{\text{train}}$ such that the Renyi-DP composed cost equals `TOTAL_EPSILON`.

In [ ]:
ALU.enable_lora_on_paths(gemma_lm.backbone, probe.selected_paths, LORA_RANK)
print(f"LoRA enabled on {len(probe.selected_paths)} layers (rank={LORA_RANK}).")

STEPS_PER_EPOCH = TRAIN_SIZE // BATCH_SIZE
TOTAL_TRAIN_STEPS = EPOCHS * STEPS_PER_EPOCH
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS

SIGMA_TRAIN = ALU.calibrate_train_sigma(
    target_eps=TOTAL_EPSILON,
    probe_sigma=probe.noise_multiplier,
    num_probe_samples=probe.n_seen,
    effective_batch_size=EFFECTIVE_BATCH_SIZE,
    train_steps=TOTAL_TRAIN_STEPS,
    train_size=TRAIN_SIZE,
    delta=DELTA,
)
composed_eps = ALU.compose_probe_and_training(
    probe_sigma=probe.noise_multiplier,
    num_probe_samples=probe.n_seen,
    train_sigma=SIGMA_TRAIN,
    effective_batch_size=EFFECTIVE_BATCH_SIZE,
    train_steps=TOTAL_TRAIN_STEPS,
    train_size=TRAIN_SIZE,
    delta=DELTA,
)
print(
    f"target total_eps={TOTAL_EPSILON} -> sigma_train={SIGMA_TRAIN:.4f}; "
    f"composed (eps, delta) ~= ({composed_eps:.4f}, {DELTA:.1e})"
)

dp_cfg = keras_api.DPKerasConfig(
    epsilon=TOTAL_EPSILON,
    delta=DELTA,
    noise_multiplier=SIGMA_TRAIN,
    clipping_norm=CLIPPING_NORM,
    batch_size=BATCH_SIZE,
    train_steps=TOTAL_TRAIN_STEPS,
    train_size=TRAIN_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    seed=SEED,
)
gemma_lm = keras_api.make_private(gemma_lm, dp_cfg)

## Compile and fine-tune

`fit()` may be called only once per `make_private` — the declared (eps, delta) budget rules out further training.

In [ ]:
optimizer = keras.optimizers.Adam(
    learning_rate=LEARNING_RATE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
)
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

gemma_lm.fit(
    x=TRAIN_DS_BATCHED,
    epochs=EPOCHS,
    validation_data=VALIDATION_DS_BATCHED,
)

print("\n=== After fine-tuning ===")
show_inference()

## Evaluate ROUGE on the test split

In [ ]:
gemma_lm.preprocessor.sequence_length = TEST_DS_SEQUENCE_LENGTH

METRIC_FNS = {
    "rouge_1": keras_hub.metrics.RougeN(order=1),
    "rouge_2": keras_hub.metrics.RougeN(order=2),
    "rouge_l": keras_hub.metrics.RougeL(),
}


def _common_prefix(a, b):
  i = 0
  while i < len(a) and i < len(b) and a[i] == b[i]:
    i += 1
  return i


for batch in tqdm.tqdm(TEST_DS_BATCHED):
  prompts = [p.decode("utf-8") for p in batch["prompts"].numpy()]
  outputs = gemma_lm.generate(prompts)
  outputs = [o[_common_prefix(p, o) :] for p, o in zip(prompts, outputs)]
  targets = [s.decode("utf-8") for s in batch["responses"].numpy()]
  for m in METRIC_FNS.values():
    m.update_state(targets, outputs)

RESULT = {k: float(m.result()["f1_score"]) for k, m in METRIC_FNS.items()}
print(RESULT)

## Results

ROUGE F1 on the test split, Gemma3-1B + LoRA (rank 64) + DP-SGD at $\varepsilon=4$, sequence length 1024. **Baseline** is `keras_hub`'s default `enable_lora()` (query + value only); **100%** enables LoRA on every attention projection (query/key/value/attention_output) in every block; **top 5%** keeps only the top-5% attention layers ranked by the DP probe.

| Dataset | Method | ROUGE-1 | ROUGE-2 | ROUGE-L |
|---|---|---|---|---|
| CNN/DailyMail | baseline (query + value, keras_hub default) | 0.217 | 0.084 | 0.157 |
| CNN/DailyMail | 100% attn projections | 0.154 | 0.046 | 0.114 |
| CNN/DailyMail | **top 5% attn (probe)** | **0.237** | **0.096** | **0.168** |
| XSum | baseline (query + value, keras_hub default) | 0.247 | 0.064 | 0.192 |
| XSum | 100% attn projections | 0.205 | 0.046 | 0.158 |
| XSum | **top 5% attn (probe)** | **0.264** | **0.073** | **0.203** |

Selecting the top 5% attention layers via the DP probe beats the `keras_hub` default on both datasets, and is dramatically better than enabling LoRA on every attention projection — fewer trainable parameters means less DP noise per useful direction. The effect is consistent across two different summarization styles (multi-sentence highlights vs single-sentence abstract).

## Citation

If you find this notebook useful in your research, please cite:

```bibtex
@article{gong2026dpsapf,
  title   = {DP-SAPF: Saliency-Aware Parameter Fine-tuning of Public Models for Differentially Private Image Synthesis},
  author  = {Chen Gong and Kecen Li and Zinan Lin and Tianhao Wang},
  journal = {arXiv preprint arXiv:2605.30312},
  year    = {2026},
  url     = {https://arxiv.org/abs/2605.30312}
}
```